## 1 setup

In [48]:
%pip install -q "datasets==3.6.0" pandas pyarrow numpy scipy scikit-learn tqdm matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
import ast
import json
import math
import re #espressioni regolari
from pathlib import Path

import joblib #salvare oggetti python
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import csr_matrix,save_npz,load_npz 
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer #trasforma testi dei prodotti in vettori numerici
from sklearn.preprocessing import normalize 
from tqdm.auto import tqdm

pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",160)
pd.set_option("display.float_format",lambda x:f"{x:,.6f}")

In [50]:
CATEGORY="Electronics"
RANDOM_STATE=42
K_VALUES=(5,10,20)
PRIMARY_METRIC="ndcg"
PRIMARY_K=10

MAX_EVAL_ROWS=None
SAVE_RANKS=True
SAVE_CONTENT_ARTIFACTS=True

MAX_TFIDF_FEATURES=20000 #20000 termini/caratteristiche testuali al max
TFIDF_MIN_DF=2 #tiene parole che compaiono in almeno due prodotti
TFIDF_MAX_DF=0.85 #elimina parole che compaiono nell 85% dei prodotti
TFIDF_NGRAM_RANGE=(1,2)  #usa sia parole singole, sia doppie
TFIDF_STOP_WORDS="english"  #rimuove parole comuni inglesi

TRAIN_SVD_FALLBACK=True 
SVD_FALLBACK_FACTORS=4
SVD_N_ITER=7

DEMO_USERS_N=3
DEMO_TOP_N=10
DEMO_SIMILAR_ITEMS_N=10

HYBRID_CONFIGS={ #combinare content-based, svd e popularity
    "Hybrid_Content70_Pop30":{"content":0.70,"svd":0.00,"popularity":0.30},
    "Hybrid_SVD70_Content30":{"content":0.30,"svd":0.70,"popularity":0.00},
    "Hybrid_SVD50_Content25_Pop25":{"content":0.25,"svd":0.50,"popularity":0.25},
    "Hybrid_SVD40_Content40_Pop20":{"content":0.40,"svd":0.40,"popularity":0.20},
}

## 2 caricamento dati

In [51]:
PROJECT_ROOT=Path("..").resolve() if (Path("..")/"data").exists() else Path(".").resolve()
RECOMMENDER_DIR=PROJECT_ROOT/"data"/"processed"/"electronics_recommender"
NOTEBOOK04_DIR=RECOMMENDER_DIR/"notebook04_models"
NOTEBOOK05_DIR=RECOMMENDER_DIR/"notebook05_content_hybrid"
NOTEBOOK05_DIR.mkdir(parents=True,exist_ok=True)

TRAIN_PATH=RECOMMENDER_DIR/"train.parquet"
VALIDATION_EVAL_PATH=RECOMMENDER_DIR/"validation_eval.parquet"
TEST_EVAL_PATH=RECOMMENDER_DIR/"test_eval.parquet"
VALIDATION_NEGATIVE_PATH=RECOMMENDER_DIR/"negative_samples_validation.parquet"
TEST_NEGATIVE_PATH=RECOMMENDER_DIR/"negative_samples_test.parquet"
ITEMS_PATH=RECOMMENDER_DIR/"items.parquet"
USER_MAPPING_PATH=RECOMMENDER_DIR/"user_mapping.parquet"
ITEM_MAPPING_PATH=RECOMMENDER_DIR/"item_mapping.parquet"
SPLIT_SUMMARY_PATH=RECOMMENDER_DIR/"split_summary.json"
QA_PATH=RECOMMENDER_DIR/"qa.parquet"

NOTEBOOK04_METRICS_PATH=NOTEBOOK04_DIR/"notebook04_metrics.csv"
NOTEBOOK04_COMPARISON_PATH=NOTEBOOK04_DIR/"notebook04_comparison.csv"
NOTEBOOK04_SUMMARY_PATH=NOTEBOOK04_DIR/"notebook04_summary.json"
POPULARITY_SCORES_PATH=NOTEBOOK04_DIR/"popularity_scores.parquet"
SVD_USER_EMB_PATH=NOTEBOOK04_DIR/"best_svd_user_embeddings.npy"
SVD_ITEM_EMB_PATH=NOTEBOOK04_DIR/"best_svd_item_embeddings.npy"
SVD_INFO_PATH=NOTEBOOK04_DIR/"best_svd_info.json"

required_paths={
    "train":TRAIN_PATH,
    "validation_eval":VALIDATION_EVAL_PATH,
    "test_eval":TEST_EVAL_PATH,
    "negative_samples_validation":VALIDATION_NEGATIVE_PATH,
    "negative_samples_test":TEST_NEGATIVE_PATH,
    "items":ITEMS_PATH,
    "user_mapping":USER_MAPPING_PATH,
    "item_mapping":ITEM_MAPPING_PATH,
}

missing_paths=[name for name,path in required_paths.items() if not path.exists()]

print("Project root:",PROJECT_ROOT)
print("Recommender dir:",RECOMMENDER_DIR)
print("Notebook04 dir:",NOTEBOOK04_DIR)
print("Notebook05 dir:",NOTEBOOK05_DIR)

for name,path in required_paths.items():
    print(f"{name}:{path} | exists={path.exists()}")

if missing_paths:
    raise FileNotFoundError("File mancanti dal Notebook 03:"+",".join(missing_paths))

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Recommender dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender
Notebook04 dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\notebook04_models
Notebook05 dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\notebook05_content_hybrid
train:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\train.parquet | exists=True
validation_eval:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\validation_eval.parquet | exists=True
test_eval:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\test_eval.parquet | exists=True
negative_samples_validation:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\negative_samples_validation.parquet | exists=True
negative_samples_test:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\negative_

In [ ]:
#file parquet principali
train_df=pd.read_parquet(TRAIN_PATH)
validation_eval_df=pd.read_parquet(VALIDATION_EVAL_PATH)
test_eval_df=pd.read_parquet(TEST_EVAL_PATH)
validation_samples_df=pd.read_parquet(VALIDATION_NEGATIVE_PATH)
test_samples_df=pd.read_parquet(TEST_NEGATIVE_PATH)
items_df=pd.read_parquet(ITEMS_PATH)
user_mapping_df=pd.read_parquet(USER_MAPPING_PATH)
item_mapping_df=pd.read_parquet(ITEM_MAPPING_PATH)

if SPLIT_SUMMARY_PATH.exists():
    split_summary=json.loads(SPLIT_SUMMARY_PATH.read_text(encoding="utf-8"))
else:
    split_summary={}

if NOTEBOOK04_SUMMARY_PATH.exists():
    notebook04_summary=json.loads(NOTEBOOK04_SUMMARY_PATH.read_text(encoding="utf-8"))
else:
    notebook04_summary={}

def parse_negative_items(value): #converte colonna negative_item_idxs in lista pulita di interi
    if value is None: #se mancante
        return []
    if isinstance(value,np.ndarray): #se array numpy, converte in lista
        value=value.tolist()
    if isinstance(value,(list,tuple,set)): #se lista, tupla o set, converte ogni el in intero
        return [int(item) for item in value]
    if isinstance(value,str): #se stringa, la pulisce, e la interpreta come oggetto python
        value=value.strip()
        if not value:
            return []
        parsed=ast.literal_eval(value)
        return [int(item) for item in parsed]
    try:
        if pd.isna(value):
            return []
    except (TypeError,ValueError):
        pass
    return [int(value)]

validation_samples_df=validation_samples_df.copy()
test_samples_df=test_samples_df.copy()
validation_samples_df["negative_item_idxs"]=validation_samples_df["negative_item_idxs"].apply(parse_negative_items) #applica funzione, per avere lista di interi
test_samples_df["negative_item_idxs"]=test_samples_df["negative_item_idxs"].apply(parse_negative_items)

if MAX_EVAL_ROWS is not None:
    validation_samples_df=validation_samples_df.head(int(MAX_EVAL_ROWS)).copy()
    test_samples_df=test_samples_df.head(int(MAX_EVAL_ROWS)).copy()

loaded_shapes=pd.DataFrame([ #riassunto dataframe caricati
    {"name":"train","rows":len(train_df),"cols":train_df.shape[1]},
    {"name":"validation_eval","rows":len(validation_eval_df),"cols":validation_eval_df.shape[1]},
    {"name":"test_eval","rows":len(test_eval_df),"cols":test_eval_df.shape[1]},
    {"name":"validation_samples","rows":len(validation_samples_df),"cols":validation_samples_df.shape[1]},
    {"name":"test_samples","rows":len(test_samples_df),"cols":test_samples_df.shape[1]},
    {"name":"items","rows":len(items_df),"cols":items_df.shape[1]},
    {"name":"user_mapping","rows":len(user_mapping_df),"cols":user_mapping_df.shape[1]},
    {"name":"item_mapping","rows":len(item_mapping_df),"cols":item_mapping_df.shape[1]},
])

display(loaded_shapes)
display(pd.DataFrame.from_dict(split_summary,orient="index",columns=["value"]).head(30))
display(pd.DataFrame.from_dict(notebook04_summary,orient="index",columns=["value"]).head(30))

,name,rows,cols
0,train,287752,14
1,validation_eval,131376,14
2,test_eval,131352,14
3,validation_samples,131376,7
4,test_samples,131352,7
5,items,12306,13
6,user_mapping,131398,2
7,item_mapping,12306,2


,value
category,Electronics
source_config,user3_item8
positive_rating_threshold,4
min_user_interactions_after,3
min_item_interactions_after,8
n_negatives,100
random_state,42
temporal_column,datetime
original_candidate_interactions,844060
positive_interactions_before_refilter,671903


,value
category,Electronics
primary_metric,ndcg
primary_k,10
best_model_validation,MostPopular
best_validation_score,0.284317
models_evaluated,"[MostPopular, ItemKNN_5, ItemKNN_10, ItemKNN_50, ItemKNN_100, SVD_2, SVD_4, SVD_6, SVD_8, SVD_10, SVD_12, SVD_16, SVD_24, SVD_32]"
train_rows,287752
validation_samples,131376
test_samples,131352
n_users,131398


## 3 costruzione del testo per ogni prodotto
metadati come nome, marca, descrizione, brand

In [53]:


if "item_idx" not in items_df.columns:
    items_df=items_df.merge(item_mapping_df,on="parent_asin",how="left")

items_df=items_df.copy()
items_df["item_idx"]=items_df["item_idx"].astype(np.int64)


def is_missing_value(value): #per vedere se un valore è mancante, None, NaN 
    if value is None:
        return True
    try:
        result=pd.isna(value)
        if isinstance(result,(bool,np.bool_)):
            return bool(result)
    except (TypeError,ValueError):
        return False
    return False


def parse_json_like(value): #converte un valore in una forma python
    if is_missing_value(value):
        return []
    if isinstance(value,(list,tuple,set,dict)):
        return value
    if isinstance(value,str): #se è stringa la pulisce dagli spazi
        text=value.strip()
        if not text or text.lower() in {"none","nan","null"}:
            return []
        try:
            return json.loads(text) #se è json valido
        except json.JSONDecodeError:
            try:
                return ast.literal_eval(text) #se quello di prima non è andato bene
            except (SyntaxError,ValueError):
                return text #se niente va, restituisce testo originale
    return value


def flatten_text(value,max_parts=40): #prende valore complesso come lista o diz e lo trasforma in una singola stringa testuale
    parsed=parse_json_like(value)
    parts=[]

    def add_part(obj): #ricorsiva
        if len(parts)>=max_parts:
            return
        if is_missing_value(obj):
            return
        if isinstance(obj,dict):  #se è dizionario, scorre chiavi e valori
            for key,val in obj.items():
                if len(parts)>=max_parts:
                    break
                if is_missing_value(val):
                    continue
                if isinstance(val,(list,tuple,set,dict)):
                    add_part(val)
                else:
                    text=str(val).strip()
                    if text:
                        parts.append(str(key)+" "+text)
        elif isinstance(obj,(list,tuple,set)): #se trova lista, tupla o set, entra dentro ogni elemento
            for item in obj:
                if len(parts)>=max_parts:
                    break
                add_part(item)
        else: #valore semplice, lo converte in testo e lo aggiunge
            text=str(obj).strip()
            if text:
                parts.append(text)

    add_part(parsed)
    return " ".join(parts) #unisce i pezzi in una stringa


def clean_text(text): #pulisce testo finale prima di darlo al TF-IDF
    text=str(text)
    text=re.sub(r"<[^>]+>"," ",text) #rimuove tag HTML
    text=re.sub(r"[^A-Za-z0-9]+"," ",text) #elimina simboli, punteggiatura, parentesi, trattini
    text=re.sub(r"\s+"," ",text) #toglie spazi multipli, e mette spazi singoli
    return text.strip().lower() 


def first_existing_value(row,columns): #prende una riga del Dataframe e una lista di colonne , poi restituisce prima valore non mancante
    for col in columns:
        if col in row.index and not is_missing_value(row[col]):
            value=str(row[col]).strip()
            if value:
                return value
    return ""

In [54]:
def build_product_text(row): #prende riga di items_df, cioè un prodotto e costruisce una stringa testuale pulita
    title=first_existing_value(row,["title"])
    category=first_existing_value(row,["category","main_category"])
    store=first_existing_value(row,["store","brand"])
    features=flatten_text(row["features_json"],max_parts=30) if "features_json" in row.index else flatten_text(row.get("features",[]),max_parts=30)
    description=flatten_text(row["description_json"],max_parts=20) if "description_json" in row.index else flatten_text(row.get("description",[]),max_parts=20)
    categories=flatten_text(row["categories_json"],max_parts=20) if "categories_json" in row.index else flatten_text(row.get("categories",[]),max_parts=20)
    details=flatten_text(row["details_json"],max_parts=12) if "details_json" in row.index else flatten_text(row.get("details",{}),max_parts=12)
    text=" ".join([title,title,category,store,features,description,categories,details]) #unisce tutto in una sola stringa
    return clean_text(text) #lo pulisce e lo ritorna

items_text_df=items_df[[col for col in items_df.columns if col in { #creiamo dataframe più piccolo
    "item_idx","parent_asin","title","category","main_category","store","brand",
    "features_json","description_json","categories_json","details_json",
    "features","description","categories","details","avg_rating","average_rating",
    "num_ratings","rating_number","price","image_url"
}]].copy()

items_text_df["content_text"]=items_text_df.apply(build_product_text,axis=1)  #applica la funzione di sopra, a ogni prodotto
items_text_df=items_text_df.sort_values("item_idx").drop_duplicates("item_idx").reset_index(drop=True) #rimuove duplicati

n_users=int(max(user_mapping_df["user_idx"].max(),train_df["user_idx"].max())+1) #n utenti
n_items=int(max(item_mapping_df["item_idx"].max(),train_df["item_idx"].max(),items_text_df["item_idx"].max())+1) #n items

content_texts=[""]*n_items #lista lunga quanto numero tot di prodotti
for row in items_text_df[["item_idx","content_text"]].itertuples(index=False): #per ogni prodotto,mette il testo nella pos corrispondente al suo item_idx
    content_texts[int(row.item_idx)]=row.content_text

empty_text_count=sum(1 for text in content_texts if not text) #conta testi vuoti
#calcoliamo quante parole ha il testo prodotto, calcoliamo media e percentili
text_length_stats=pd.Series([len(text.split()) for text in content_texts],name="tokens_per_item").describe(percentiles=[0.25,0.5,0.75,0.9,0.95])

print("n_users:",n_users)
print("n_items:",n_items)
print("prodotti con testo vuoto:",empty_text_count)
display(text_length_stats.to_frame())
display(items_text_df[["item_idx","parent_asin","title","content_text"]].head(5))

n_users: 131398
n_items: 12306
prodotti con testo vuoto: 0


,tokens_per_item
count,"12,306.000000"
mean,352.240452
std,255.119731
min,23.000000
25%,184.000000
50%,288.000000
75%,454.000000
90%,649.000000
95%,787.000000
max,"5,974.000000"


,item_idx,parent_asin,title,content_text
0,0,0972683275,"VideoSecu 24"" Long Arm TV Wall Mount Low Profile Articulating Full Motion Swing Tilt Wall Bracket for Most 22"" to 55"" LED LCD UHD TV Monitor Flat Screen VES...",videosecu 24 long arm tv wall mount low profile articulating full motion swing tilt wall bracket for most 22 to 55 led lcd uhd tv monitor flat screen vesa 2...
1,1,1400501466,"Barnes & Noble NOOK Tablet 16gb (Color, BNTV250)",barnes noble nook tablet 16gb color bntv250 barnes noble nook tablet 16gb color bntv250 computers barnes noble nook color tablettm nov 2011 bntv250 hd movie...
2,2,1400532655,"Barnes & Noble NOOK Color 7"" Tablet",barnes noble nook color 7 tablet barnes noble nook color 7 tablet all electronics barnes noble get social surf the web play games even listen to music over ...
3,3,1400532736,NOOK Simple Touch,nook simple touch nook simple touch all electronics barnes noble 6 inch touch screen quick downloads with wi fi easy navigation share reading with friends m...
4,4,1400599997,Barnes & Noble NOOK ebook reader (WiFi + 3G)[B&W],barnes noble nook ebook reader wifi 3g b w barnes noble nook ebook reader wifi 3g b w all electronics barnes noble barnes noble brand ebook reader nook e in...


## 4 rappresentazione TF-IDF dei prodotti
La rappresentazione TF-IDF assegna peso maggiore ai termini utili a distinguere un prodotto dagli altri


In [55]:
tfidf_vectorizer=TfidfVectorizer( #configuriamo traformatore TF-IDF
    max_features=MAX_TFIDF_FEATURES, 
    min_df=TFIDF_MIN_DF,
    max_df=TFIDF_MAX_DF,
    ngram_range=TFIDF_NGRAM_RANGE,
    stop_words=TFIDF_STOP_WORDS,
    strip_accents="unicode",
    lowercase=True,
    dtype=np.float32,
)

try:
    #impara vocabolario dei testi dei prodotti, trasforma ogni prodotto in un vettore TF-IDF
    item_tfidf_matrix=tfidf_vectorizer.fit_transform(content_texts).tocsr() 
except ValueError as error: 
    #se non va, riprova con filtro meno severo, accetta anche parole presenti in un solo prodotto
    print("TF-IDF fallback con min_df=1:",error)
    tfidf_vectorizer.set_params(min_df=1)
    item_tfidf_matrix=tfidf_vectorizer.fit_transform(content_texts).tocsr()
#normalizza ogni riga della matrice, portati sulla stessa scala
item_tfidf_matrix=normalize(item_tfidf_matrix,norm="l2",axis=1,copy=False)
#salva i nomi delle parole/bigrammi usati
feature_names=np.array(tfidf_vectorizer.get_feature_names_out())

tfidf_stats={
    "items":int(item_tfidf_matrix.shape[0]),
    "features":int(item_tfidf_matrix.shape[1]), #num di termini TF-IDF usati, cioè colonne
    "non_zero_values":int(item_tfidf_matrix.nnz),
    "density_pct":float(item_tfidf_matrix.nnz/(item_tfidf_matrix.shape[0]*item_tfidf_matrix.shape[1])*100),
}

display(pd.DataFrame.from_dict(tfidf_stats,orient="index",columns=["value"]))
print("Prime feature:",feature_names[:30].tolist())

,value
items,"12,306.000000"
features,"20,000.000000"
non_zero_values,"2,244,999.000000"
density_pct,0.912156


Prime feature: ['00', '00 00', '000', '000 000', '000 contrast', '000 dot', '000 dots', '000 dynamic', '000 hours', '000 hz', '000 movies', '000 photo', '000 photos', '000 songs', '000 systems', '00001', '000hz', '001', '002', '003', '004', '004g', '006', '008g', '01', '01 batteries', '01 inches', '01 ounces', '01 pounds', '010']


In [56]:
def top_terms_for_item(item_idx,top_n=12): #prende un prodotto tramite indice e restituisce i suoi termini TF-IDF più importanti
    row=item_tfidf_matrix[int(item_idx)]
    if row.nnz==0:
        return []
    data=row.data # i pesi 
    indices=row.indices #gli indici delle feature
    order=np.argsort(-data)[:int(top_n)]
    return [(str(feature_names[indices[pos]]),float(data[pos])) for pos in order] #lista di coppie (termine, peso)

term_preview_rows=[]
for item_idx in items_text_df["item_idx"].head(5).astype(int).tolist(): #primi 5 prodotti di items
    term_preview_rows.append({
        "item_idx":int(item_idx),
        "parent_asin":items_text_df.loc[items_text_df["item_idx"]==item_idx,"parent_asin"].iloc[0],
        "title":items_text_df.loc[items_text_df["item_idx"]==item_idx,"title"].iloc[0] if "title" in items_text_df.columns else "",
        "top_terms":top_terms_for_item(item_idx,12),
    })

display(pd.DataFrame(term_preview_rows))

,item_idx,parent_asin,title,top_terms
0,0,0972683275,"VideoSecu 24"" Long Arm TV Wall Mount Low Profile Articulating Full Motion Swing Tilt Wall Bracket for Most 22"" to 55"" LED LCD UHD TV Monitor Flat Screen VES...","[(vesa, 0.3312319815158844), (tv, 0.24706195294857025), (videosecu, 0.23686696588993073), (wall, 0.1901065856218338), (tilt, 0.17753304541110992), (55, 0.16..."
1,1,1400501466,"Barnes & Noble NOOK Tablet 16gb (Color, BNTV250)","[(nook, 0.49399855732917786), (barnes noble, 0.23119249939918518), (barnes, 0.2300475388765335), (noble, 0.22583161294460297), (noble nook, 0.17716658115386..."
2,2,1400532655,"Barnes & Noble NOOK Color 7"" Tablet","[(barnes noble, 0.32357701659202576), (barnes, 0.3219745457172394), (noble, 0.3160739243030548), (books, 0.2804471552371979), (reading, 0.2598644495010376),..."
3,3,1400532736,NOOK Simple Touch,"[(barnes noble, 0.261881947517395), (ebook readers, 0.261881947517395), (barnes, 0.2605850100517273), (noble, 0.255809485912323), (simple touch, 0.252593606..."
4,4,1400599997,Barnes & Noble NOOK ebook reader (WiFi + 3G)[B&W],"[(barnes noble, 0.3807520866394043), (barnes, 0.37886643409729004), (noble, 0.3719232380390167), (ebook, 0.3643423318862915), (nook, 0.30132174491882324), (..."


## 5 matrice utenti-prodotti e profili content-based
profilo di ogni utente creato sommando i vettori TF-IDF dei prodotti con cui ha interagito nel training

In [57]:
train_user_idx=train_df["user_idx"].astype(np.int64).to_numpy()
train_item_idx=train_df["item_idx"].astype(np.int64).to_numpy()
train_values=np.ones(len(train_df),dtype=np.float32) #1 per ogni interazione

#matrice sparsa train n_users x n_items
train_matrix=csr_matrix((train_values,(train_user_idx,train_item_idx)),shape=(n_users,n_items),dtype=np.float32)
train_matrix.sum_duplicates()
train_matrix.eliminate_zeros()

train_seen_by_user=( #per ogni utente salva insieme dei prodotti nel train
    train_df.groupby("user_idx")["item_idx"]
    .apply(lambda values:set(values.astype(int)))
    .to_dict()
)
#unisce tutte le interazioni note
all_interactions_for_seen=pd.concat([train_df,validation_eval_df,test_eval_df],ignore_index=True)
all_seen_by_user=(
    all_interactions_for_seen.groupby("user_idx")["item_idx"]
    .apply(lambda values:set(values.astype(int)))
    .to_dict()
)
#elenco prodotti che compaiono almeno una volta nel train
train_item_candidates=np.array(sorted(train_df["item_idx"].astype(int).unique()),dtype=np.int64)

matrix_stats={
    "n_users":int(n_users),
    "n_items":int(n_items),
    "train_interactions":int(train_matrix.nnz),
    "train_density_pct":float(train_matrix.nnz/(n_users*n_items)*100),
    "train_item_candidates":int(len(train_item_candidates)),
}

display(pd.DataFrame.from_dict(matrix_stats,orient="index",columns=["value"]))

,value
n_users,"131,398.000000"
n_items,"12,306.000000"
train_interactions,"287,752.000000"
train_density_pct,0.017796
train_item_candidates,"12,299.000000"


In [58]:
user_profile_matrix=(train_matrix@item_tfidf_matrix).tocsr() 
#matrice utenti x prodotti e matrice prodotti x terminiTF-IDF
#facciamo prodotto matriciale e ottieniamo utenti x termini TF-IDF

#normalizza ogni profilo utente
user_profile_matrix=normalize(user_profile_matrix,norm="l2",axis=1,copy=False)

profile_stats={
    "user_profiles_shape":str(user_profile_matrix.shape),
    "user_profile_non_zero_values":int(user_profile_matrix.nnz),
    "users_with_non_empty_profile":int(np.diff(user_profile_matrix.indptr).astype(bool).sum()),
}

display(pd.DataFrame.from_dict(profile_stats,orient="index",columns=["value"]))

,value
user_profiles_shape,"(131398, 20000)"
user_profile_non_zero_values,47609835
users_with_non_empty_profile,131398


## 6 funzioni di valutazione

In [ ]:
#calcola posizione del prodotto positivo nella lista dei candidati
def rank_positive(user_idx,positive_item_idx,negative_item_idxs,score_fn): #riceve utente, prodotto pos, lista prodotti neg, funzione di scoring
    candidate_items=[int(positive_item_idx)]+[int(item) for item in negative_item_idxs] #lista candidati pos e neg
    scores=score_fn(int(user_idx),candidate_items) #score per ogni prodotto
    scores=np.asarray(scores,dtype=np.float64) #converte in array numpy di float64
    scores=np.nan_to_num(scores,nan=-np.inf,posinf=np.inf,neginf=-np.inf) #sistema valori anomali
    positive_score=float(scores[0]) #salva score del positivo
    rank=1
    for item_idx,score in zip(candidate_items[1:],scores[1:]): #scorre, escludendo positivo, e se neg ha score piu alto, positivo scende di una posizione
        if score>positive_score:
            rank+=1
        elif score==positive_score and int(item_idx)<int(positive_item_idx):
            rank+=1
    return int(rank),positive_score

In [ ]:
#prende tutti i rank dei prodotti positivi e calcola le metriche per diversi valori di k
def calculate_topk_metrics(ranks,k_values): #riceve lista dei rank dei positivi e valori di k
    ranks_array=np.asarray(ranks,dtype=np.float64)
    rows=[]
    for k in k_values:
        hits=ranks_array<=int(k) #true se fra i primi k
        hitrate=float(np.mean(hits)) if len(ranks_array) else np.nan
        ndcg_values=np.where(hits,1.0/np.log2(ranks_array+1.0),0.0)
        mrr_values=np.where(hits,1.0/ranks_array,0.0)
        rows.append({
            "K":int(k),
            "hitrate":hitrate,
            "recall":hitrate,
            "ndcg":float(np.mean(ndcg_values)) if len(ranks_array) else np.nan,
            "mrr":float(np.mean(mrr_values)) if len(ranks_array) else np.nan,
            "n_examples":int(len(ranks_array)),
        })
    return pd.DataFrame(rows)

In [ ]:
#funzione di valutazione
def evaluate_model(samples_df,model_name,score_fn,split_name,k_values=K_VALUES):
    #riceve tabella con pos e neg, nome modello, funzione dei punteggi, 
    ranks=[]
    rank_rows=[]
    iterator=samples_df.itertuples(index=False) #crea iteratore sulle righe di samplesdf
    for row in tqdm(iterator,total=len(samples_df),desc=f"Evaluate {model_name} {split_name}"): 
        #calcola il rank del positivo
        rank,positive_score=rank_positive(row.user_idx,row.positive_item_idx,row.negative_item_idxs,score_fn)
        ranks.append(rank)
        rank_rows.append({
            "model":model_name,
            "split":split_name,
            "user_idx":int(row.user_idx),
            "positive_item_idx":int(row.positive_item_idx),
            "positive_rank":int(rank),
            "positive_score":float(positive_score),
            "num_negatives":int(len(row.negative_item_idxs)),
        })
    metrics_df=calculate_topk_metrics(ranks,k_values) #calcola metriche aggregate
    metrics_df.insert(0,"split",split_name) #inserisce nella tabella
    metrics_df.insert(0,"model",model_name)
    ranks_df=pd.DataFrame(rank_rows)
    return metrics_df,ranks_df

## 7 modello content-based TF-IDF
punteggio contentbased è la similarità tra profilo testuale dell'utente e vettore testuale del prodotto candidato

In [ ]:
def score_content_tfidf(user_idx,item_idxs):
    #riceve utente da valutare e lista dei prodotti candidati
    item_idxs=np.asarray(item_idxs,dtype=np.int64)
    user_vector=user_profile_matrix[int(user_idx)] #recupera profilo contentbased dell'utente
    if user_vector.nnz==0 or len(item_idxs)==0: 
        return np.zeros(len(item_idxs),dtype=np.float64)
    item_vectors=item_tfidf_matrix[item_idxs] #recupera vettori tfidf dei prodotti candidati
    scores=item_vectors.dot(user_vector.T) #calcola similarità tra ogni prodotto candidato e il profilo dell'utente, prodotto scalare
    #prodotto scalare corrisponde a similarità coseno, essendo normalizzati i vettori
    return np.asarray(scores.toarray()).ravel().astype(np.float64)


#valutazione del modelli ContentIDF
content_validation_metrics,content_validation_ranks=evaluate_model(
    validation_samples_df,
    "ContentTFIDF",
    score_content_tfidf,
    "validation",
)

display(content_validation_metrics)

Evaluate ContentTFIDF validation: 100%|██████████| 131376/131376 [00:29<00:00, 4521.48it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,ContentTFIDF,validation,5,0.191458,0.191458,0.137711,0.120060,131376
1,ContentTFIDF,validation,10,0.274023,0.274023,0.164241,0.130911,131376
2,ContentTFIDF,validation,20,0.399441,0.399441,0.195723,0.139433,131376


## 8 recupero segnali collaborativi notebook4

In [63]:
if NOTEBOOK04_METRICS_PATH.exists():
    previous_metrics_df=pd.read_csv(NOTEBOOK04_METRICS_PATH)
    previous_metrics_df["source"]="notebook04"
else:
    previous_metrics_df=pd.DataFrame()

if NOTEBOOK04_COMPARISON_PATH.exists():
    previous_comparison_df=pd.read_csv(NOTEBOOK04_COMPARISON_PATH)
else:
    previous_comparison_df=pd.DataFrame()

print("Metriche Notebook 04 disponibili:",not previous_metrics_df.empty)
if not previous_metrics_df.empty:
    display(previous_metrics_df.sort_values(["split","K","ndcg"],ascending=[True,True,False]).head(20))

Metriche Notebook 04 disponibili: True


,model,split,K,hitrate,recall,ndcg,mrr,n_examples,source
42,MostPopular,test,5,0.361639,0.361639,0.261726,0.228795,131352,notebook04
60,SVD_4,test,5,0.355267,0.355267,0.259025,0.227282,131352,notebook04
63,SVD_6,test,5,0.354132,0.354132,0.256577,0.224405,131352,notebook04
66,SVD_8,test,5,0.349816,0.349816,0.253429,0.221642,131352,notebook04
57,SVD_2,test,5,0.345126,0.345126,0.251190,0.220232,131352,notebook04
69,SVD_10,test,5,0.345225,0.345225,0.249529,0.217965,131352,notebook04
72,SVD_12,test,5,0.338761,0.338761,0.243823,0.212503,131352,notebook04
75,SVD_16,test,5,0.330456,0.330456,0.237607,0.206987,131352,notebook04
78,SVD_24,test,5,0.307296,0.307296,0.219698,0.190808,131352,notebook04
81,SVD_32,test,5,0.294453,0.294453,0.209614,0.181673,131352,notebook04


In [64]:
if POPULARITY_SCORES_PATH.exists():
    popularity_scores_df=pd.read_parquet(POPULARITY_SCORES_PATH)
    popularity_scores={int(row.item_idx):float(row.popularity_score) for row in popularity_scores_df.itertuples(index=False)}
else:
    popularity_series=train_df["item_idx"].value_counts()
    popularity_scores={int(item_idx):float(count) for item_idx,count in popularity_series.items()}
    popularity_scores_df=pd.DataFrame({"item_idx":list(popularity_scores.keys()),"popularity_score":list(popularity_scores.values())})


def score_popularity(user_idx,item_idxs): #non usa utente, essendo baseline non personalizzata
    return np.array([popularity_scores.get(int(item_idx),0.0) for item_idx in item_idxs],dtype=np.float64)

popularity_summary={
    "items_with_popularity":int(len(popularity_scores)),
    "max_popularity":float(max(popularity_scores.values())) if popularity_scores else 0.0,
    "mean_popularity":float(np.mean(list(popularity_scores.values()))) if popularity_scores else 0.0,
}

display(pd.DataFrame.from_dict(popularity_summary,orient="index",columns=["value"]))

,value
items_with_popularity,"12,299.000000"
max_popularity,"1,118.000000"
mean_popularity,23.396374


In [65]:
svd_available=False
svd_user_embeddings=None
svd_item_embeddings=None
svd_info={}

if SVD_USER_EMB_PATH.exists() and SVD_ITEM_EMB_PATH.exists(): #carica svd
    svd_user_embeddings=np.load(SVD_USER_EMB_PATH)
    svd_item_embeddings=np.load(SVD_ITEM_EMB_PATH)
    svd_info=json.loads(SVD_INFO_PATH.read_text(encoding="utf-8")) if SVD_INFO_PATH.exists() else {"model":"SVD_from_notebook04"}
    svd_available=True
elif TRAIN_SVD_FALLBACK and n_users>1 and n_items>1: #se file svd non esistono, ne addestra uno di emergenza
    max_components=max(1,min(n_users,n_items)-1)
    actual_components=int(min(SVD_FALLBACK_FACTORS,max_components))
    svd=TruncatedSVD(n_components=actual_components,n_iter=SVD_N_ITER,random_state=RANDOM_STATE)
    svd_user_embeddings=svd.fit_transform(train_matrix).astype(np.float32)
    svd_item_embeddings=svd.components_.T.astype(np.float32)
    svd_info={
        "model":"SVD_fallback",
        "requested_components":int(SVD_FALLBACK_FACTORS),
        "actual_components":int(actual_components),
        "explained_variance_ratio":float(svd.explained_variance_ratio_.sum()),
    }
    svd_available=True


def score_svd(user_idx,item_idxs): #assegna score svd a una lista di prodotti candidati per un certo utente
    if not svd_available:
        return np.zeros(len(item_idxs),dtype=np.float64)
    user_vector=svd_user_embeddings[int(user_idx)]
    item_matrix=svd_item_embeddings[np.asarray(item_idxs,dtype=np.int64)]
    return (item_matrix@user_vector).astype(np.float64)

print("SVD disponibile:",svd_available)
display(pd.DataFrame.from_dict(svd_info,orient="index",columns=["value"]) if svd_info else pd.DataFrame())

SVD disponibile: True


,value
model,SVD_4
requested_components,4
actual_components,4
explained_variance_ratio,0.013917


## 9 modelli ibridi semplici
score finale= pesocontent * content + pesosvd * svd + pesopopolarità * popolarità

In [ ]:
def minmax_normalize_scores(scores): #porta gli score nell'intervallo 0-1
    scores=np.asarray(scores,dtype=np.float64)
    if len(scores)==0:
        return scores
    scores=np.nan_to_num(scores,nan=0.0,posinf=0.0,neginf=0.0) #toglie valori problematici
    min_score=float(np.min(scores)) 
    max_score=float(np.max(scores))
    if not np.isfinite(min_score) or not np.isfinite(max_score) or max_score<=min_score: #controlla se normalizzazione è impossibile
        return np.zeros(len(scores),dtype=np.float64)
    return (scores-min_score)/(max_score-min_score) #score normalizzato = (score-minimo)/(massimo-minimo), sempre tra 0 e 1


def make_hybrid_score_fn(weights): #crea funzione di scoring ibrida a partire da un dizionario di pesi
    content_weight=float(weights.get("content",0.0)) #peso componente contentbased
    svd_weight=float(weights.get("svd",0.0)) #peso svd
    popularity_weight=float(weights.get("popularity",0.0)) #peso popolarità

    def score_hybrid(user_idx,item_idxs): #per un utente e una lista di prodotti candidati, calcola score finale
        content_scores=minmax_normalize_scores(score_content_tfidf(user_idx,item_idxs)) #calcola score del modello contentTFIDF e li normalizza
        popularity_component=minmax_normalize_scores(score_popularity(user_idx,item_idxs)) #score di popolarità e normalizza
        if svd_weight>0 and svd_available:
            svd_component=minmax_normalize_scores(score_svd(user_idx,item_idxs)) #se si usa svd, calcola score e normalizza
        else: 
            svd_component=np.zeros(len(item_idxs),dtype=np.float64)
        final_scores=( #somma con pesi
            content_weight*content_scores
            +svd_weight*svd_component
            +popularity_weight*popularity_component
        )
        return final_scores

    return score_hybrid
#diz dei modelli da valutare
notebook05_score_functions={"ContentTFIDF":score_content_tfidf}
#scorre tutte le configurazioni ibride
for model_name,weights in HYBRID_CONFIGS.items():
    if weights.get("svd",0.0)>0 and not svd_available:
        print("Salto",model_name,"perché SVD non è disponibile")
        continue
    notebook05_score_functions[model_name]=make_hybrid_score_fn(weights) #crea funzione di scoring con quei pesi e la aggiunge al dizionario dei modelli

print("Modelli Notebook 05:",list(notebook05_score_functions.keys()))

Modelli Notebook 05: ['ContentTFIDF', 'Hybrid_Content70_Pop30', 'Hybrid_SVD70_Content30', 'Hybrid_SVD50_Content25_Pop25', 'Hybrid_SVD40_Content40_Pop20']


In [ ]:
#partiamo dai risultati già calcolati per contentTFIDF
validation_metric_frames=[content_validation_metrics]
validation_rank_frames=[content_validation_ranks]
#scorri tutti i modelli disponibili nel dizionario
for model_name,score_fn in notebook05_score_functions.items(): #scorre tutti i modelli disponibili
    if model_name=="ContentTFIDF":
        continue #lo salta, già valutato prima
    metrics_df,ranks_df=evaluate_model(validation_samples_df,model_name,score_fn,"validation")  #valuta modello corrente su validation
    validation_metric_frames.append(metrics_df) #metriche aggregate
    validation_rank_frames.append(ranks_df) #rank dettagliati esempio per esempio
    display(metrics_df)

notebook05_validation_metrics_df=pd.concat(validation_metric_frames,ignore_index=True)
notebook05_validation_ranks_df=pd.concat(validation_rank_frames,ignore_index=True)
notebook05_validation_metrics_df["source"]="notebook05"

display(notebook05_validation_metrics_df.sort_values(["K","ndcg"],ascending=[True,False]))

Evaluate Hybrid_Content70_Pop30 validation: 100%|██████████| 131376/131376 [00:35<00:00, 3665.18it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,Hybrid_Content70_Pop30,validation,5,0.297459,0.297459,0.198346,0.165889,131376
1,Hybrid_Content70_Pop30,validation,10,0.428906,0.428906,0.240815,0.183394,131376
2,Hybrid_Content70_Pop30,validation,20,0.576688,0.576688,0.278124,0.193605,131376


Evaluate Hybrid_SVD70_Content30 validation: 100%|██████████| 131376/131376 [00:39<00:00, 3289.74it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,Hybrid_SVD70_Content30,validation,5,0.359746,0.359746,0.256360,0.222278,131376
1,Hybrid_SVD70_Content30,validation,10,0.477119,0.477119,0.294289,0.237915,131376
2,Hybrid_SVD70_Content30,validation,20,0.613042,0.613042,0.328552,0.247266,131376


Evaluate Hybrid_SVD50_Content25_Pop25 validation: 100%|██████████| 131376/131376 [00:39<00:00, 3329.21it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,Hybrid_SVD50_Content25_Pop25,validation,5,0.370014,0.370014,0.261936,0.226333,131376
1,Hybrid_SVD50_Content25_Pop25,validation,10,0.495989,0.495989,0.302617,0.243089,131376
2,Hybrid_SVD50_Content25_Pop25,validation,20,0.636768,0.636768,0.338160,0.252818,131376


Evaluate Hybrid_SVD40_Content40_Pop20 validation: 100%|██████████| 131376/131376 [00:39<00:00, 3364.89it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,Hybrid_SVD40_Content40_Pop20,validation,5,0.365234,0.365234,0.263904,0.230463,131376
1,Hybrid_SVD40_Content40_Pop20,validation,10,0.483026,0.483026,0.301921,0.246110,131376
2,Hybrid_SVD40_Content40_Pop20,validation,20,0.620669,0.620669,0.336629,0.255587,131376


,model,split,K,hitrate,recall,ndcg,mrr,n_examples,source
12,Hybrid_SVD40_Content40_Pop20,validation,5,0.365234,0.365234,0.263904,0.230463,131376,notebook05
9,Hybrid_SVD50_Content25_Pop25,validation,5,0.370014,0.370014,0.261936,0.226333,131376,notebook05
6,Hybrid_SVD70_Content30,validation,5,0.359746,0.359746,0.256360,0.222278,131376,notebook05
3,Hybrid_Content70_Pop30,validation,5,0.297459,0.297459,0.198346,0.165889,131376,notebook05
0,ContentTFIDF,validation,5,0.191458,0.191458,0.137711,0.120060,131376,notebook05
10,Hybrid_SVD50_Content25_Pop25,validation,10,0.495989,0.495989,0.302617,0.243089,131376,notebook05
13,Hybrid_SVD40_Content40_Pop20,validation,10,0.483026,0.483026,0.301921,0.246110,131376,notebook05
7,Hybrid_SVD70_Content30,validation,10,0.477119,0.477119,0.294289,0.237915,131376,notebook05
4,Hybrid_Content70_Pop30,validation,10,0.428906,0.428906,0.240815,0.183394,131376,notebook05
1,ContentTFIDF,validation,10,0.274023,0.274023,0.164241,0.130911,131376,notebook05
